# LeetCode 866 — Prime Palindrome bằng Python

Link đề: https://leetcode.com/problems/prime-palindrome/ · Độ khó: **Medium** · Chủ đề: Math (toán), Number Theory (lý thuyết số), Primality Test (kiểm tra nguyên tố)

Skeleton LeetCode:

```python
class Solution:
    def primePalindrome(self, n: int) -> int:
        ...
```

Ghi chú: tìm **số nguyên tố đối xứng** (palindromic prime) nhỏ nhất `>= n`. Duyệt từng số tới đáp án có thể phải đi hàng chục triệu bước (ví dụ `n = 10^7` có đáp án `100030001`), nên cần hai nhận xét toán học: **số đối xứng độ dài chẵn đều chia hết cho 11** (chỉ `11` là số nguyên tố) và có thể **sinh trực tiếp** các số đối xứng từ **nửa trái** thay vì duyệt mọi số.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm số nhỏ nhất `>= n` vừa là số nguyên tố vừa là số đối xứng (đọc xuôi ngược như nhau); đề đảm bảo đáp án tồn tại trong `[2, 2 * 10^8]`.
2. Nắm các công cụ Python cần thiết: đảo ngược số bằng `divmod` hoặc chuỗi `str(x)[::-1]`, kiểm tra nguyên tố bằng chia thử tới `sqrt`, dựng chuỗi đối xứng từ nửa trái (`s + s[::-1]`, `s + s[-2::-1]`), `bisect_left` trên bảng đã sắp xếp.
3. Chứng minh và dùng tính chất: **số đối xứng có số chữ số chẵn chia hết cho 11**; do đó chỉ cần xét độ dài lẻ (và riêng số `11`).
4. Đi qua 5 cách: duyệt tuần tự, duyệt có nhảy qua khoảng vô ích, sinh đối xứng từ nửa trái kèm chia thử, sinh đối xứng kèm Miller-Rabin, bảng các số nguyên tố đối xứng tính sẵn kèm `bisect`; đối chiếu bằng oracle độc lập (mọi số đối xứng, cả độ dài chẵn).
5. Tránh các lỗi kinh điển: coi `1` là nguyên tố, vòng kiểm tra nguyên tố dùng `<` thay vì `<=` (bỏ sót số chính phương như `9`), bỏ hẳn độ dài chẵn (mất `11`), không xử lý `n` trong `(10^7, 10^8)`, và sinh nửa trái sai (thiếu chữ số giữa).

## 2. Đề bài & ví dụ

Cho số nguyên `n`, trả về **số nguyên tố đối xứng nhỏ nhất lớn hơn hoặc bằng `n`**.

Một số nguyên là **nguyên tố** nếu có đúng hai ước là `1` và chính nó (chú ý `1` **không** phải số nguyên tố). Ví dụ `2, 3, 5, 7, 11, 13` là nguyên tố.

Một số nguyên là **đối xứng** (palindrome) nếu đọc từ trái sang phải giống đọc từ phải sang trái. Ví dụ `101` và `12321`.

Test được tạo sao cho đáp án luôn tồn tại và nằm trong `[2, 2 * 10^8]`.

**Ví dụ 1:** `n = 6` -> `7`

**Ví dụ 2:** `n = 8` -> `11`

**Ví dụ 3:** `n = 13` -> `101`

**Ràng buộc:**

- `1 <= n <= 10^8`

In [1]:
from typing import List
from bisect import bisect_left
import random, time

examples = [
    (6, 7),
    (8, 11),
    (13, 101),
]
for n, expected in examples:
    print(f"n={n} kỳ vọng={expected}")

n=6 kỳ vọng=7
n=8 kỳ vọng=11
n=13 kỳ vọng=101


## 3. Kiến thức nền

### 3.1. Số đối xứng và cách đảo số

Có hai cách đảo số: (a) bằng chuỗi `int(str(x)[::-1])`; (b) bằng phép toán số học: lấy từng chữ số cuối bằng `divmod(x, 10)` và ghép vào `rev = rev * 10 + chữ_số`. Một số `x` đối xứng khi `reverse(x) == x`. Chú ý số có chữ số 0 ở cuối (như `10`) khi đảo được `1` (≠ `10`), nên không đối xứng.

In [2]:
def reverse_num(x: int) -> int:
    rev = 0
    while x:
        x, d = divmod(x, 10)
        rev = rev * 10 + d
    return rev

for x in (121, 123, 10, 1001, 7, 12321):
    print(f"{x:6} đảo = {reverse_num(x):6} đối xứng? {reverse_num(x) == x} | bằng chuỗi: {str(x) == str(x)[::-1]}")

   121 đảo =    121 đối xứng? True | bằng chuỗi: True
   123 đảo =    321 đối xứng? False | bằng chuỗi: False
    10 đảo =      1 đối xứng? False | bằng chuỗi: False
  1001 đảo =   1001 đối xứng? True | bằng chuỗi: True
     7 đảo =      7 đối xứng? True | bằng chuỗi: True
 12321 đảo =  12321 đối xứng? True | bằng chuỗi: True


### 3.2. Kiểm tra nguyên tố bằng chia thử

`x >= 2` là nguyên tố nếu không chia hết cho mọi `v` trong `2 .. sqrt(x)`. Điều kiện vòng lặp đúng là `v * v <= x` (**có dấu bằng**): với `x = 9` thì `v = 3` cần được thử; nếu dùng `<` thì `9` bị coi nguyên tố.

In [3]:
def is_prime(x: int) -> bool:
    if x < 2:
        return False
    v = 2
    while v * v <= x:
        if x % v == 0:
            return False
        v += 1
    return True

print([x for x in range(1, 40) if is_prime(x)])
print("9 (đúng):", is_prime(9), "| 49:", is_prime(49), "| 1:", is_prime(1), "| 2:", is_prime(2))

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37]
9 (đúng): False | 49: False | 1: False | 2: True


### 3.3. Số đối xứng độ dài chẵn chia hết cho 11

Số `x = a_1 a_2 ... a_k a_k ... a_2 a_1` (độ dài `2k`). Quy tắc chia hết cho 11: hiệu tổng chữ số ở vị trí lẻ và vị trí chẵn chia hết cho 11. Với số đối xứng độ dài chẵn, chữ số ở vị trí `i` và `2k + 1 - i` bằng nhau nhưng **khác tính chẵn lẻ vị trí** (hai vị trí cộng lại `2k + 1` là lẻ), nên hiệu bằng `0`. Vậy mọi số đối xứng độ dài chẵn đều chia hết cho 11, và chỉ `11` là nguyên tố.

In [4]:
def palindromes_of_length(L):
    half = (L + 1) // 2
    for h in range(10 ** (half - 1), 10 ** half):
        s = str(h)
        yield int(s + (s[::-1] if L % 2 == 0 else s[-2::-1]))

for L in (2, 4, 6):
    pals = list(palindromes_of_length(L))
    assert all(x % 11 == 0 for x in pals)
    print(f"độ dài {L}: {len(pals)} số đối xứng, tất cả chia hết cho 11; ví dụ {pals[:4]}")
primes_even = [x for L in (2, 4, 6) for x in palindromes_of_length(L) if is_prime(x)]
print("số nguyên tố đối xứng độ dài chẵn (<= 6 chữ số):", primes_even)

độ dài 2: 9 số đối xứng, tất cả chia hết cho 11; ví dụ [11, 22, 33, 44]
độ dài 4: 90 số đối xứng, tất cả chia hết cho 11; ví dụ [1001, 1111, 1221, 1331]
độ dài 6: 900 số đối xứng, tất cả chia hết cho 11; ví dụ [100001, 101101, 102201, 103301]
số nguyên tố đối xứng độ dài chẵn (<= 6 chữ số): [11]


### 3.4. Sinh số đối xứng từ nửa trái

Một số đối xứng độ dài `L` hoàn toàn xác định bởi **nửa trái** gồm `(L + 1) // 2` chữ số. Độ dài lẻ: lấy chữ số giữa một lần (`s + s[-2::-1]`); độ dài chẵn: `s + s[::-1]`. Duyệt nửa trái tăng dần cho ra các số đối xứng **tăng dần** (trong cùng độ dài).

In [5]:
for h in (12, 13, 99):
    s = str(h)
    print(f"nửa trái {s}: độ dài chẵn -> {s + s[::-1]} | độ dài lẻ (bỏ chữ số cuối khi lật) -> {s + s[-2::-1]}")
print(list(palindromes_of_length(3))[:12], "...", len(list(palindromes_of_length(3))), "số")

nửa trái 12: độ dài chẵn -> 1221 | độ dài lẻ (bỏ chữ số cuối khi lật) -> 121
nửa trái 13: độ dài chẵn -> 1331 | độ dài lẻ (bỏ chữ số cuối khi lật) -> 131
nửa trái 99: độ dài chẵn -> 9999 | độ dài lẻ (bỏ chữ số cuối khi lật) -> 999
[101, 111, 121, 131, 141, 151, 161, 171, 181, 191, 202, 212] ... 90 số


### 3.5. Miller-Rabin (kiểm tra nguyên tố xác suất, ở đây là tất định)

Chia thử tới `sqrt(x)` với `x ~ 2 * 10^8` cần tới `~14 000` phép chia cho mỗi số nguyên tố. Miller-Rabin kiểm tra nguyên tố trong `O(log x)` phép nhân modulo; với `x < 3.4 * 10^14` chỉ cần các cơ số `2, 3, 5, 7, 11, 13, 17` để **luôn đúng** (tất định). Ta dùng nó ở Cách 4.

In [6]:
def is_prime_mr(x: int) -> bool:
    if x < 2:
        return False
    small = (2, 3, 5, 7, 11, 13, 17)
    for p in small:
        if x % p == 0:
            return x == p
    d, s = x - 1, 0
    while d % 2 == 0:
        d //= 2
        s += 1
    for a in small:
        y = pow(a, d, x)
        if y in (1, x - 1):
            continue
        for _ in range(s - 1):
            y = y * y % x
            if y == x - 1:
                break
        else:
            return False
    return True

assert all(is_prime_mr(x) == is_prime(x) for x in range(0, 20000))
print("Miller-Rabin khớp chia thử trên [0, 20000); ví dụ:", is_prime_mr(100030001), is_prime_mr(10 ** 8 + 7), is_prime_mr(10 ** 8 + 1))

Miller-Rabin khớp chia thử trên [0, 20000); ví dụ: True True False


## 4. Ý tưởng nền tảng / trực giác (tính tay)

`n = 13`. Xét lần lượt: `13` (không đối xứng), `14`, ..., `22` (đối xứng nhưng `22 = 2 * 11`), `33`, `44`, ..., `99` (đều chia hết cho 11, hợp số), `101`: đối xứng và nguyên tố ⇒ `101`.

Rút ra: khi qua các số 2 chữ số `22..99` ta chỉ phí công; sau `11` mọi số đối xứng 2 chữ số là hợp số. Tương tự mọi số đối xứng 4, 6, 8 chữ số là hợp số. Sau `11` số nguyên tố đối xứng tiếp theo là `101` (3 chữ số), rồi `131, 151, 181, 191, 313, ...`.

In [7]:
n = 13
x = n
steps = 0
while True:
    steps += 1
    if reverse_num(x) == x:
        print(f"{x}: đối xứng; nguyên tố? {is_prime(x)}")
        if is_prime(x):
            break
    x += 1
print("kết quả", x, "sau", steps, "số đã xét")

22: đối xứng; nguyên tố? False
33: đối xứng; nguyên tố? False
44: đối xứng; nguyên tố? False
55: đối xứng; nguyên tố? False
66: đối xứng; nguyên tố? False
77: đối xứng; nguyên tố? False
88: đối xứng; nguyên tố? False
99: đối xứng; nguyên tố? False
101: đối xứng; nguyên tố? True
kết quả 101 sau 89 số đã xét


## 5. Cách 1 — Brute Force: duyệt tuần tự

**Ý tưởng.** Từ `n`, tăng dần; với mỗi `x` kiểm tra đối xứng (`reverse(x) == x`) rồi nguyên tố; dừng ở số đầu tiên thỏa. Đúng nhưng có thể rất chậm.

In [8]:
def prime_pal_brute(n: int) -> int:
    x = n
    while True:
        if reverse_num(x) == x and is_prime(x):
            return x
        x += 1

for n, expected in examples:
    got = prime_pal_brute(n)
    print(n, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

6 -> 7 OK
8 -> 11 OK
13 -> 101 OK


**Độ phức tạp và nút thắt.** Số bước bằng `đáp án - n + 1`. Với `n = 10^7` đáp án là `100030001`: gần `9 * 10^7` bước, mỗi bước đảo số `O(log)`: không thể. Hãy đếm số bước (chỉ đếm, không chạy hết với `n` lớn):

In [9]:
pals_check = lambda x: reverse_num(x) == x
print(f"{'n':>10} {'số bước tới đáp án':>20}")
for n in (1, 10, 100, 1000, 10 ** 4, 10 ** 5, 10 ** 6):
    ans = prime_pal_brute(n)
    print(f"{n:>10} {ans - n + 1:>20,}   (đáp án {ans})")
print("n = 10^7 cần khoảng", f"{100030001 - 10 ** 7 + 1:,}", "bước: quá lớn")

         n   số bước tới đáp án
         1                    2   (đáp án 2)
        10                    2   (đáp án 11)
       100                    2   (đáp án 101)
      1000                9,302   (đáp án 10301)
     10000                  302   (đáp án 10301)


    100000              903,002   (đáp án 1003001)
   1000000                3,002   (đáp án 1003001)
n = 10^7 cần khoảng 90,030,002 bước: quá lớn


Nút thắt: ta xét **rất nhiều số không đối xứng** (đa số số trong khoảng). Chỉ cần xét các số đối xứng, mà chúng có thể **sinh trực tiếp**.

## 6. Ý tưởng thuật toán / bước đột phá

Hai ý tưởng:

1. **Chặn khoảng vô ích (Cách 2).** Mọi số đối xứng có 8 chữ số (`10^7 .. 10^8 - 1`) là hợp số (độ dài chẵn, chia hết cho 11). Vậy nếu `n` rơi vào `(10^7, 10^8)` thì nhảy thẳng tới `10^8` (số 9 chữ số đối xứng đầu tiên là `100000001`).
2. **Sinh trực tiếp các số đối xứng (Cách 3-5).** Duyệt **nửa trái** tăng dần để sinh số đối xứng tăng dần; chỉ xét độ dài lẻ (và `11`). Chỉ có khoảng `10^4` nửa trái cho 9 chữ số (`10^4 .. 10^5 - 1`) thay vì `10^8` số.

| Cách | Ý tưởng |
|---|---|
| 2 | Duyệt tuần tự nhưng nhảy qua `(10^7, 10^8)` |
| 3 | Sinh đối xứng từ nửa trái, chia thử tới `sqrt` |
| 4 | Như Cách 3 nhưng dùng Miller-Rabin |
| 5 | Bảng mọi số nguyên tố đối xứng `<= 2 * 10^8`, tra bằng `bisect_left` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Duyệt tuần tự có nhảy

**Giải thích từng đoạn.** Giống Cách 1, thêm: nếu `10^7 < n < 10^8` thì `n = 10^8` (đặt trước khi tăng). Đây là lời giải chuẩn của doocs.

In [10]:
def prime_pal_skip(n: int) -> int:
    while True:
        if reverse_num(n) == n and is_prime(n):
            return n
        if 10 ** 7 < n < 10 ** 8:
            n = 10 ** 8
        n += 1

for n, expected in examples:
    assert prime_pal_skip(n) == expected
print("duyệt có nhảy OK")

duyệt có nhảy OK


**Code nộp LeetCode (Cách 2):**

In [11]:
class Solution:
    def primePalindrome(self, n: int) -> int:
        def is_prime(x):
            if x < 2:
                return False
            v = 2
            while v * v <= x:
                if x % v == 0:
                    return False
                v += 1
            return True

        def reverse(x):
            res = 0
            while x:
                res = res * 10 + x % 10
                x //= 10
            return res

        while 1:
            if reverse(n) == n and is_prime(n):
                return n
            if 10**7 < n < 10**8:
                n = 10**8
            n += 1

sol_skip = Solution().primePalindrome
print([sol_skip(n) for n, _ in examples])

[7, 11, 101]


**Độ phức tạp (Cách 2):** số số đối xứng phải xét vẫn phải duyệt qua mọi số trong khoảng (tối đa vài triệu bước với `n <= 10^7`), mỗi bước `O(log n)` cho đảo số; mỗi số đối xứng thử nguyên tố `O(sqrt x)`. Chạy được nhưng chậm (vài giây).

---

### 7.2. Cách 3 — Sinh số đối xứng từ nửa trái

**Ý tưởng khác.** Với mỗi độ dài `L` từ `len(str(n))` trở lên: nếu `L` chẵn và `L != 2` thì **bỏ qua** (luôn hợp số). Ngược lại duyệt nửa trái `h` từ `10^(half-1)` tới `10^half - 1`, dựng số đối xứng, nếu `>= n` và nguyên tố thì trả về ngay (vì tăng dần).

**Giải thích từng đoạn.**

- `half = (L + 1) // 2`.
- Dựng `pal = int(s + s[-2::-1])` (độ dài lẻ) hoặc `int(s + s[::-1])` (độ dài 2).
- Độ dài 1 (`half = 1`): `s[-2::-1]` rỗng nên `pal = h` (các chữ số `1..9`).
- Giới hạn độ dài tối đa 9 (`2 * 10^8` có 9 chữ số).

In [12]:
def prime_pal_generate(n: int) -> int:
    for L in range(len(str(n)), 10):
        if L % 2 == 0 and L != 2:
            continue                                     # mọi đối xứng độ dài chẵn (trừ 11) chia hết cho 11
        half = (L + 1) // 2
        for h in range(10 ** (half - 1), 10 ** half):
            s = str(h)
            pal = int(s + (s[::-1] if L % 2 == 0 else s[-2::-1]))
            if pal >= n and is_prime(pal):
                return pal
    raise ValueError("không có đáp án trong phạm vi")

for n, expected in examples:
    assert prime_pal_generate(n) == expected
print("sinh nửa trái OK")

sinh nửa trái OK


**Code nộp LeetCode (Cách 3):**

In [13]:
class Solution:
    def primePalindrome(self, n: int) -> int:
        def is_prime(x: int) -> bool:
            if x < 2:
                return False
            v = 2
            while v * v <= x:
                if x % v == 0:
                    return False
                v += 1
            return True

        for L in range(len(str(n)), 10):
            if L % 2 == 0 and L != 2:
                continue
            half = (L + 1) // 2
            for h in range(10 ** (half - 1), 10 ** half):
                s = str(h)
                pal = int(s + (s[::-1] if L % 2 == 0 else s[-2::-1]))
                if pal >= n and is_prime(pal):
                    return pal
        return -1

sol_generate = Solution().primePalindrome
print([sol_generate(n) for n, _ in examples])

[7, 11, 101]


**Độ phức tạp (Cách 3):** tối đa `~10^4` nửa trái cho 9 chữ số và `~10^3` cho 7 chữ số; mỗi số đối xứng thử chia `O(sqrt x)` (tới `~14 000` phép). Tổng cỡ `10^4 * 10^4 = 10^8` phép chia trong tình huống xấu nhất (`n` gần `10^8`), nhưng hầu hết bị loại nhanh bởi các ước nhỏ; thực tế chỉ cần vài chục ms tới ~1 s.

---

### 7.3. Cách 4 — Sinh nửa trái + Miller-Rabin

**Ý tưởng khác.** Thay phép chia thử `O(sqrt x)` bằng Miller-Rabin tất định `O(log x)`. Giữ nguyên bộ sinh. Tổng nhanh hơn nhiều.

In [14]:
def prime_pal_mr(n: int) -> int:
    for L in range(len(str(n)), 10):
        if L % 2 == 0 and L != 2:
            continue
        half = (L + 1) // 2
        for h in range(10 ** (half - 1), 10 ** half):
            s = str(h)
            pal = int(s + (s[::-1] if L % 2 == 0 else s[-2::-1]))
            if pal >= n and is_prime_mr(pal):
                return pal
    raise ValueError("không có đáp án trong phạm vi")

for n, expected in examples:
    assert prime_pal_mr(n) == expected
print("sinh nửa trái + Miller-Rabin OK")

sinh nửa trái + Miller-Rabin OK


**Code nộp LeetCode (Cách 4):**

In [15]:
class Solution:
    def primePalindrome(self, n: int) -> int:
        def is_prime(x: int) -> bool:
            if x < 2:
                return False
            small = (2, 3, 5, 7, 11, 13, 17)
            for p in small:
                if x % p == 0:
                    return x == p
            d, s = x - 1, 0
            while d % 2 == 0:
                d //= 2
                s += 1
            for a in small:
                y = pow(a, d, x)
                if y in (1, x - 1):
                    continue
                for _ in range(s - 1):
                    y = y * y % x
                    if y == x - 1:
                        break
                else:
                    return False
            return True

        for L in range(len(str(n)), 10):
            if L % 2 == 0 and L != 2:
                continue
            half = (L + 1) // 2
            for h in range(10 ** (half - 1), 10 ** half):
                s = str(h)
                pal = int(s + (s[::-1] if L % 2 == 0 else s[-2::-1]))
                if pal >= n and is_prime(pal):
                    return pal
        return -1

sol_mr = Solution().primePalindrome
print([sol_mr(n) for n, _ in examples])

[7, 11, 101]


**Độ phức tạp (Cách 4):** `O(số nửa trái * log x)`: khoảng `10^5 * 30` phép; chạy dưới 0,2 giây.

---

### 7.4. Cách 5 — Bảng tính sẵn và `bisect`

**Ý tưởng khác.** Chỉ có **vài trăm** số nguyên tố đối xứng `<= 2 * 10^8`. Tính sẵn (bằng bộ sinh + Miller-Rabin) và lưu trong một danh sách đã sắp xếp; với mỗi `n` dùng `bisect_left` tìm số đầu tiên `>= n`.

**Giải thích từng đoạn.** Bảng là kết quả của việc duyệt mọi độ dài (1, 2, 3, 5, 7, 9) rồi gom mọi số nguyên tố đối xứng `<= 2 * 10^8`. Bảng chỉ tính một lần.

In [16]:
def build_table(limit=2 * 10 ** 8):
    table = []
    for L in range(1, 10):
        if L % 2 == 0 and L != 2:
            continue
        half = (L + 1) // 2
        for h in range(10 ** (half - 1), 10 ** half):
            s = str(h)
            pal = int(s + (s[::-1] if L % 2 == 0 else s[-2::-1]))
            if pal <= limit and is_prime_mr(pal):
                table.append(pal)
    return table

TABLE = build_table()
print("số nguyên tố đối xứng <= 2*10^8:", len(TABLE), "| 12 số đầu:", TABLE[:12], "| số cuối:", TABLE[-1])

def prime_pal_table(n: int) -> int:
    return TABLE[bisect_left(TABLE, n)]

for n, expected in examples:
    assert prime_pal_table(n) == expected
print("bảng + bisect OK")

số nguyên tố đối xứng <= 2*10^8: 2205 | 12 số đầu: [2, 3, 5, 7, 11, 101, 131, 151, 181, 191, 313, 353] | số cuối: 199999991
bảng + bisect OK


**Code nộp LeetCode (Cách 5):** tính bảng một lần ở mức lớp (hoặc ngay trong hàm, vì chỉ mất khoảng 0,1 giây).

In [17]:
class Solution:
    _table = None

    def primePalindrome(self, n: int) -> int:
        from bisect import bisect_left
        cls = type(self)
        if cls._table is None:
            def is_prime(x: int) -> bool:
                if x < 2:
                    return False
                small = (2, 3, 5, 7, 11, 13, 17)
                for p in small:
                    if x % p == 0:
                        return x == p
                d, s = x - 1, 0
                while d % 2 == 0:
                    d //= 2
                    s += 1
                for a in small:
                    y = pow(a, d, x)
                    if y in (1, x - 1):
                        continue
                    for _ in range(s - 1):
                        y = y * y % x
                        if y == x - 1:
                            break
                    else:
                        return False
                return True

            table = []
            for L in range(1, 10):
                if L % 2 == 0 and L != 2:
                    continue
                half = (L + 1) // 2
                for h in range(10 ** (half - 1), 10 ** half):
                    s = str(h)
                    pal = int(s + (s[::-1] if L % 2 == 0 else s[-2::-1]))
                    if pal <= 2 * 10 ** 8 and is_prime(pal):
                        table.append(pal)
            cls._table = table
        return cls._table[bisect_left(cls._table, n)]

sol_table = Solution().primePalindrome
print([sol_table(n) for n, _ in examples])

[7, 11, 101]


**Độ phức tạp (Cách 5):** dựng bảng `O(số nửa trái * log x)` một lần (cỡ `10^5` Miller-Rabin); mỗi truy vấn `O(log |bảng|)`. Bộ nhớ `O(|bảng|)` với `|bảng|` cỡ vài trăm.

## 8. TRACE từng bước

### 8.1. Cách 3 cho `n = 13`

`L = 2` (chữ số của 13): độ dài 2 cho phép (chỉ `11` là nguyên tố). Nửa trái `h = 10..99`: `11, 22, ..., 99`; mọi số `< 13` bị loại bởi `pal >= n` (11 < 13) và `22..99` chia hết cho 11 nên không nguyên tố. Sang `L = 3` (chẵn `4` bị bỏ qua nên L = 3): nửa trái `10..99`: `101` (từ `h = 10`) đối xứng, nguyên tố ⇒ trả `101`.

In [18]:
def trace_generate(n):
    for L in range(len(str(n)), 10):
        if L % 2 == 0 and L != 2:
            print(f"độ dài {L} chẵn: bỏ qua (chia hết cho 11)")
            continue
        half = (L + 1) // 2
        print(f"độ dài {L}: nửa trái {10 ** (half - 1)}..{10 ** half - 1}")
        shown = 0
        for h in range(10 ** (half - 1), 10 ** half):
            s = str(h)
            pal = int(s + (s[::-1] if L % 2 == 0 else s[-2::-1]))
            ok_n = pal >= n
            prime = is_prime(pal) if ok_n else None
            if shown < 12 or (ok_n and prime):
                print(f"    h={h}: pal={pal} >= n? {ok_n}; nguyên tố? {prime}")
                shown += 1
            if ok_n and prime:
                print("  => kết quả", pal)
                return pal

assert trace_generate(13) == 101

độ dài 2: nửa trái 1..9
    h=1: pal=11 >= n? False; nguyên tố? None
    h=2: pal=22 >= n? True; nguyên tố? False
    h=3: pal=33 >= n? True; nguyên tố? False
    h=4: pal=44 >= n? True; nguyên tố? False
    h=5: pal=55 >= n? True; nguyên tố? False
    h=6: pal=66 >= n? True; nguyên tố? False
    h=7: pal=77 >= n? True; nguyên tố? False
    h=8: pal=88 >= n? True; nguyên tố? False
    h=9: pal=99 >= n? True; nguyên tố? False
độ dài 3: nửa trái 10..99
    h=10: pal=101 >= n? True; nguyên tố? True
  => kết quả 101


### 8.2. Cách 2 cho `n = 8`

Duyệt `8`: đối xứng nhưng `8` không nguyên tố; `9`: không nguyên tố; `10`: không đối xứng; `11`: đối xứng và nguyên tố ⇒ `11`.

In [19]:
x = 8
while True:
    sym, pr = reverse_num(x) == x, is_prime(x)
    print(f"x={x}: đối xứng={sym}, nguyên tố={pr}")
    if sym and pr:
        break
    x += 1
print("kết quả", x)
assert x == 11

x=8: đối xứng=True, nguyên tố=False
x=9: đối xứng=True, nguyên tố=False
x=10: đối xứng=False, nguyên tố=False
x=11: đối xứng=True, nguyên tố=True
kết quả 11


### 8.3. Cách 4: Miller-Rabin trên một số cụ thể

Kiểm tra `x = 101`: `x - 1 = 100 = 25 * 2^2` (`d = 25`, `s = 2`). Với cơ số `a`, tính `y = a^d mod x` và bình phương liên tiếp; `101` nguyên tố nên mọi cơ số đều qua.

In [20]:
def trace_mr(x):
    small = (2, 3, 5, 7, 11, 13, 17)
    for p in small:
        if x % p == 0:
            print(f"{x} chia hết {p}: {'là chính nó (nguyên tố)' if x == p else 'hợp số'}")
            return x == p
    d, s = x - 1, 0
    while d % 2 == 0:
        d //= 2
        s += 1
    print(f"x = {x}: x - 1 = {d} * 2^{s}")
    for a in small:
        y = pow(a, d, x)
        seq = [y]
        ok = y in (1, x - 1)
        for _ in range(s - 1):
            if ok:
                break
            y = y * y % x
            seq.append(y)
            if y == x - 1:
                ok = True
        print(f"  cơ số {a:2}: dãy {seq} -> {'qua' if ok else 'KHÔNG qua (hợp số)'}")
        if not ok:
            return False
    return True

assert trace_mr(101) is True
print()
assert trace_mr(1000001) is False

x = 101: x - 1 = 25 * 2^2
  cơ số  2: dãy [10, 100] -> qua
  cơ số  3: dãy [10, 100] -> qua
  cơ số  5: dãy [1] -> qua
  cơ số  7: dãy [10, 100] -> qua
  cơ số 11: dãy [10, 100] -> qua
  cơ số 13: dãy [100] -> qua
  cơ số 17: dãy [100] -> qua

x = 1000001: x - 1 = 15625 * 2^6
  cơ số  2: dãy [729331, 175638, 676196, 573176, 398446, 56157] -> KHÔNG qua (hợp số)


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (sinh nửa trái + chia thử): ngắn, đúng, chạy nhanh (dưới vài trăm mili-giây trong thực tế), không cần kiến thức Miller-Rabin. Cách 4/5 nhanh hơn nữa.

```python
class Solution:
    def primePalindrome(self, n: int) -> int:
        def is_prime(x: int) -> bool:
            if x < 2:
                return False
            v = 2
            while v * v <= x:
                if x % v == 0:
                    return False
                v += 1
            return True

        for L in range(len(str(n)), 10):
            if L % 2 == 0 and L != 2:
                continue
            half = (L + 1) // 2
            for h in range(10 ** (half - 1), 10 ** half):
                s = str(h)
                pal = int(s + (s[::-1] if L % 2 == 0 else s[-2::-1]))
                if pal >= n and is_prime(pal):
                    return pal
        return -1
```

In [21]:
class Solution:
    def primePalindrome(self, n: int) -> int:
        def is_prime(x: int) -> bool:
            if x < 2:
                return False
            v = 2
            while v * v <= x:
                if x % v == 0:
                    return False
                v += 1
            return True

        for L in range(len(str(n)), 10):
            if L % 2 == 0 and L != 2:
                continue
            half = (L + 1) // 2
            for h in range(10 ** (half - 1), 10 ** half):
                s = str(h)
                pal = int(s + (s[::-1] if L % 2 == 0 else s[-2::-1]))
                if pal >= n and is_prime(pal):
                    return pal
        return -1

sol = Solution()
for n, expected in examples:
    got = sol.primePalindrome(n)
    print(n, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

6 -> 7 OK
8 -> 11 OK
13 -> 101 OK


## 10. Vì sao đúng

**(a) Số đối xứng độ dài chẵn chia hết cho 11.** Xem mục 3.3. Gọi chữ số `d_1 d_2 ... d_{2k}` với `d_i = d_{2k+1-i}`. Tổng xen kẽ `S = d_1 - d_2 + d_3 - ... - d_{2k}` (vị trí lẻ dương, chẵn âm). Cặp `(d_i, d_{2k+1-i})` có `i` và `2k + 1 - i` khác tính chẵn lẻ (tổng lẻ), nên một cái mang dấu `+`, cái kia mang dấu `-`, triệt tiêu: `S = 0`, chia hết cho 11 theo dấu hiệu chia hết cho 11. Hệ quả: mọi số đối xứng độ dài chẵn lớn hơn `11` là hợp số (có ước 11 và nhỏ hơn chính nó), còn `11` là nguyên tố.

**(b) Bộ sinh đúng thứ tự và đầy đủ.** Mỗi số đối xứng độ dài `L` ứng với đúng một nửa trái `h` gồm `(L + 1) // 2` chữ số (chữ số đầu khác 0). Khi `h` tăng thì số đối xứng tăng (so sánh từ chữ số đầu). Duyệt độ dài tăng dần rồi `h` tăng dần cho ra mọi số đối xứng theo thứ tự tăng; số đầu tiên `>= n` thỏa nguyên tố là đáp án. Vì chỉ bỏ qua độ dài chẵn (trừ 2) mà chúng toàn hợp số (a), ta không bỏ sót đáp án.

**(c) Phạm vi.** Đề đảm bảo đáp án `<= 2 * 10^8` (có tối đa 9 chữ số), nên vòng `L` tới 9 là đủ.

**(d) Cách 2 chính xác.** Khoảng `(10^7, 10^8)` chỉ gồm số 8 chữ số; số đối xứng trong đó có độ dài chẵn nên hợp số; nhảy tới `10^8` không bỏ sót. (`n = 10^7` chính nó là 8 chữ số, ta kiểm tra riêng ở nhánh đầu của vòng lặp vì điều kiện nhảy là `10^7 < n`; `10^7` không đối xứng.)

**(e) Miller-Rabin tất định.** Với `x < 3 317 044 064 679 887 385 961 981` các cơ số 2..37 đủ; với `x < 3,4 * 10^14` các cơ số `2, 3, 5, 7, 11, 13, 17` đủ. Ta kiểm tra thực nghiệm trên mọi số tới `2 * 10^6` bằng chia thử (và tin vào định lý cho phạm vi lớn).

**Kiểm chứng bằng máy:** oracle độc lập sinh **mọi** số đối xứng (độ dài chẵn lẫn lẻ) tới `2 * 10^8`, lọc nguyên tố bằng chia thử; sau đó so mọi cách với oracle trên **mọi** `n` từ 1 tới `2 * 10^5` và trên nhiều `n` ngẫu nhiên tới `10^8`.

In [22]:
def all_palindromes_upto(limit):
    out = []
    for L in range(1, len(str(limit)) + 1):
        half = (L + 1) // 2
        for h in range(10 ** (half - 1), 10 ** half):
            s = str(h)
            pal = int(s + (s[::-1] if L % 2 == 0 else s[-2::-1]))
            if pal <= limit:
                out.append(pal)
    return sorted(out)

oracle_table = [x for x in all_palindromes_upto(2 * 10 ** 8) if is_prime(x)]
print("oracle: số nguyên tố đối xứng <= 2*10^8:", len(oracle_table), "| khớp bảng Cách 5:", oracle_table == TABLE)
assert oracle_table == TABLE

def oracle(n: int) -> int:
    return oracle_table[bisect_left(oracle_table, n)]

solutions = {
    "generate": prime_pal_generate,
    "mr": prime_pal_mr,
    "table": prime_pal_table,
    "sol_generate": sol_generate,
    "sol_mr": sol_mr,
    "sol_table": sol_table,
    "final": Solution().primePalindrome,
}

# mọi n trong [1, 2*10^5]: các cách sinh/bảng khớp oracle; hai cách duyệt tuần tự chỉ kiểm tra n <= 3000 và các điểm chọn lọc
for n in range(1, 200001):
    ref = oracle(n)
    assert prime_pal_table(n) == ref
    if n % 97 == 0 or n < 2000:
        for name, fn in solutions.items():
            assert fn(n) == ref, (name, n, fn(n), ref)
for n in range(1, 3000):
    assert prime_pal_brute(n) == oracle(n)
    assert prime_pal_skip(n) == oracle(n)
print("n = 1..2*10^5 (bảng) và các mẫu (mọi cách sinh): khớp oracle; duyệt tuần tự khớp với n < 3000")

oracle: số nguyên tố đối xứng <= 2*10^8: 2205 | khớp bảng Cách 5: True


n = 1..2*10^5 (bảng) và các mẫu (mọi cách sinh): khớp oracle; duyệt tuần tự khớp với n < 3000


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt tuần tự | `O((đáp án - n) * log n + số đối xứng * sqrt)` — tới `~10^8` bước | `O(1)` |
| 2. Duyệt có nhảy | như Cách 1 nhưng chặn `(10^7, 10^8)` | `O(1)` |
| 3. Sinh nửa trái + chia thử | `O(10^4 * sqrt(x))` tệ nhất | `O(1)` |
| 4. Sinh nửa trái + Miller-Rabin | `O(10^5 * log x)` | `O(1)` |
| 5. Bảng + `bisect` | dựng `O(10^5 * log x)` một lần; truy vấn `O(log 781)` | `O(781)` |

Đo thời gian trên các `n` điển hình (những `n` khó cho duyệt tuần tự như `10^7` chỉ chạy với cách sinh/bảng):

In [23]:
tests = [1, 6, 13, 100, 9989900, 10 ** 6, 10 ** 7, 10 ** 7 + 1, 99999999, 10 ** 8, 7654321, 12345678]
ref = [oracle(n) for n in tests]
for name in ["generate", "mr", "table"]:
    fn = solutions[name]
    t0 = time.perf_counter()
    res = [fn(n) for n in tests]
    dt = (time.perf_counter() - t0) * 1000
    assert res == ref, name
    print(f"{name:9} {len(tests)} truy vấn: {dt:9.1f} ms")
print("kết quả:", ref)
t0 = time.perf_counter()
prime_pal_skip(9989900)
print(f"duyệt có nhảy cho n = 9989900 (đáp án 100030001): {1000 * (time.perf_counter() - t0):.0f} ms")

generate  12 truy vấn:      16.0 ms
mr        12 truy vấn:       8.4 ms
table     12 truy vấn:       0.0 ms
kết quả: [2, 7, 101, 101, 100030001, 1003001, 100030001, 100030001, 100030001, 100030001, 7654567, 100030001]
duyệt có nhảy cho n = 9989900 (đáp án 100030001): 49 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai, đối chiếu mọi cách với oracle độc lập và test ngẫu nhiên có seed (`1 <= n <= 10^8`).

In [24]:
# 12.1 ví dụ đề
for n, expected in examples:
    for name, fn in solutions.items():
        assert fn(n) == expected, (name, n)
    assert prime_pal_brute(n) == prime_pal_skip(n) == expected
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    (1, 2),                    # 1 không nguyên tố -> 2
    (2, 2),                    # chính n
    (3, 3),
    (4, 5),
    (5, 5),
    (7, 7),
    (8, 11),                   # 8, 9, 10 không thỏa; 11 là nguyên tố đối xứng độ dài chẵn duy nhất
    (11, 11),
    (12, 101),                 # 22, 33, ..., 99 hợp số
    (100, 101),
    (102, 131),
    (999, 10301),              # 1001, 1111, ... (độ dài chẵn) bị loại; 10301 là số kế tiếp
    (9989900, 100030001),      # trong (10^7, 10^8) không có đáp án -> 100030001
    (10 ** 7, 100030001),
    (10 ** 8, 100030001),      # n = 10^8: đáp án 100030001
    (100030001, 100030001),
    (100030002, 100050001),
]
for n, expected in edge:
    assert oracle(n) == expected, ("oracle khác kỳ vọng", n, expected, oracle(n))
    for name, fn in solutions.items():
        assert fn(n) == expected, (name, n, fn(n), expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 17 ca


In [25]:
# 12.3 phản ví dụ: các cách sai
def is_prime_strict(x):                  # dùng v * v < x thay vì <=
    if x < 2:
        return False
    v = 2
    while v * v < x:
        if x % v == 0:
            return False
        v += 1
    return True

def wrong_one_is_prime(n):               # coi 1 là nguyên tố (bỏ kiểm tra x < 2)
    def isp(x):
        v = 2
        while v * v <= x:
            if x % v == 0:
                return False
            v += 1
        return True
    x = n
    while True:
        if str(x) == str(x)[::-1] and isp(x):
            return x
        x += 1

def wrong_strict_sqrt(n):                # kiểm tra nguyên tố bỏ sót số chính phương của số nguyên tố
    x = n
    while True:
        if str(x) == str(x)[::-1] and is_prime_strict(x):
            return x
        x += 1

def wrong_skip_all_even(n):              # bỏ mọi độ dài chẵn, kể cả 11
    for L in range(len(str(n)), 10):
        if L % 2 == 0:
            continue
        half = (L + 1) // 2
        for h in range(10 ** (half - 1), 10 ** half):
            s = str(h)
            pal = int(s + s[-2::-1])
            if pal >= n and is_prime(pal):
                return pal

def wrong_missing_middle(n):             # dựng đối xứng độ dài lẻ sai: s + s[::-1] (độ dài chẵn) cho mọi L
    for L in range(len(str(n)), 10):
        half = (L + 1) // 2
        for h in range(10 ** (half - 1), 10 ** half):
            s = str(h)
            pal = int(s + s[::-1])
            if pal >= n and is_prime(pal):
                return pal

wrongs = [
    ("coi 1 là nguyên tố", wrong_one_is_prime, 1),
    ("v * v < x (bỏ sót số chính phương)", wrong_strict_sqrt, 8),
    ("bỏ mọi độ dài chẵn (mất 11)", wrong_skip_all_even, 8),
    ("dựng đối xứng thiếu chữ số giữa", wrong_missing_middle, 12),
]
for name, fn, n in wrongs:
    truth = oracle(n)
    got = fn(n)
    print(f"{name:38} n={n}: sai = {got}, đúng = {truth}")
    assert got != truth, name
print("12.3 phản ví dụ: OK (các cách sai đều bị lộ)")

coi 1 là nguyên tố                     n=1: sai = 1, đúng = 2
v * v < x (bỏ sót số chính phương)     n=8: sai = 9, đúng = 11
bỏ mọi độ dài chẵn (mất 11)            n=8: sai = 101, đúng = 11


dựng đối xứng thiếu chữ số giữa        n=12: sai = None, đúng = 101
12.3 phản ví dụ: OK (các cách sai đều bị lộ)


In [26]:
# 12.4 ngẫu nhiên có seed, đúng ràng buộc 1 <= n <= 10^8
rng = random.Random(2866)
ns = [rng.randint(1, 10 ** 8) for _ in range(300)]
ns += [rng.randint(1, 10 ** 4) for _ in range(300)] + [rng.randint(10 ** 7, 10 ** 8) for _ in range(300)]
ns += [10 ** k for k in range(0, 9)] + [10 ** k - 1 for k in range(1, 9)] + [10 ** k + 1 for k in range(0, 8)]
for n in ns:
    ref = oracle(n)
    for name, fn in solutions.items():
        assert fn(n) == ref, (name, n, fn(n), ref)
print("12.4 ngẫu nhiên và các mốc 10^k: OK (", len(ns), "giá trị n )")

# 12.5 duyệt tuần tự (Cách 2) khớp oracle trên các n nhỏ ngẫu nhiên và gần các mốc (chạy chậm với n lớn nên chọn n tới 10^6)
rng = random.Random(5)
for n in [rng.randint(1, 10 ** 6) for _ in range(40)] + [10 ** 6 - 2, 998, 999, 1000, 9999]:
    assert prime_pal_skip(n) == oracle(n), n
print("12.5 duyệt tuần tự có nhảy khớp oracle trên n <= 10^6: OK")

# 12.6 tính chất: kết quả >= n, đối xứng, nguyên tố, và không có số nguyên tố đối xứng nào nằm giữa n và kết quả
rng = random.Random(6)
for _ in range(300):
    n = rng.randint(1, 10 ** 8)
    ans = Solution().primePalindrome(n)
    assert ans >= n and str(ans) == str(ans)[::-1] and is_prime_mr(ans)
    i = bisect_left(oracle_table, n)
    assert oracle_table[i] == ans and (i == 0 or oracle_table[i - 1] < n)
print("12.6 tính chất của đáp án: OK")

12.4 ngẫu nhiên và các mốc 10^k: OK ( 925 giá trị n )


12.5 duyệt tuần tự có nhảy khớp oracle trên n <= 10^6: OK


12.6 tính chất của đáp án: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Coi `1` là số nguyên tố

`1` chỉ có một ước nên **không** là nguyên tố; với `n = 1` đáp án là `2`.

In [27]:
print("sai:", wrong_one_is_prime(1), "| đúng:", Solution().primePalindrome(1))
assert wrong_one_is_prime(1) == 1 and Solution().primePalindrome(1) == 2

sai: 1 | đúng: 2


Vì sao sai: định nghĩa nguyên tố đòi đúng hai ước phân biệt. **Đúng:** `if x < 2: return False`.

### Lỗi 2 — Vòng kiểm tra nguyên tố dùng `v * v < x`

Với `x = 9` cần thử `v = 3` (`3 * 3 == 9`); `<` bỏ sót nên `9` bị coi là nguyên tố.

In [28]:
print("is_prime_strict(9) =", is_prime_strict(9), "| is_prime(9) =", is_prime(9))
print("sai:", wrong_strict_sqrt(8), "| đúng:", Solution().primePalindrome(8))
assert is_prime_strict(9) and wrong_strict_sqrt(8) == 9 and Solution().primePalindrome(8) == 11

is_prime_strict(9) = True | is_prime(9) = False
sai: 9 | đúng: 11


Vì sao sai: `9` đối xứng và bị coi nguyên tố nên trả `9` thay cho `11`. **Đúng:** `v * v <= x`.

### Lỗi 3 — Bỏ mọi độ dài chẵn, kể cả `11`

Mọi đối xứng chẵn chia hết cho 11 nhưng **`11` chính là số nguyên tố**.

In [29]:
print("sai (n = 8):", wrong_skip_all_even(8), "| đúng:", Solution().primePalindrome(8))
assert wrong_skip_all_even(8) == 101 and Solution().primePalindrome(8) == 11

sai (n = 8): 101 | đúng: 11


Vì sao sai: bỏ `11` làm `n = 8, 9, 10, 11` cho `101`. **Đúng:** `if L % 2 == 0 and L != 2: continue`.

### Lỗi 4 — Sinh đối xứng thiếu chữ số giữa

Độ dài lẻ phải dùng `s + s[-2::-1]` (chữ số cuối của nửa trái không lặp), không phải `s + s[::-1]`.

In [30]:
print("sai (n = 12):", wrong_missing_middle(12), "| đúng:", Solution().primePalindrome(12))
assert wrong_missing_middle(12) != Solution().primePalindrome(12)
for s in ("123", "10"):
    print(f"nửa trái {s}: lẻ đúng = {s + s[-2::-1]} | sai (s + s[::-1]) = {s + s[::-1]}")

sai (n = 12): None | đúng: 101
nửa trái 123: lẻ đúng = 12321 | sai (s + s[::-1]) = 123321
nửa trái 10: lẻ đúng = 101 | sai (s + s[::-1]) = 1001


Vì sao sai: `s + s[::-1]` luôn cho độ dài chẵn; số đối xứng độ dài lẻ không được sinh ra. **Đúng:** `s + s[-2::-1]` cho độ dài `2 * len(s) - 1`.

### Lỗi 5 — Duyệt tuần tự không xử lý khoảng `(10^7, 10^8)`

Với `n` trong khoảng đó ta duyệt hàng chục triệu số (vài phút trong Python).

In [31]:
n = 10 ** 7 + 5
steps = 100030001 - n + 1
print(f"n = {n}: duyệt tuần tự cần {steps:,} bước (mỗi bước đảo số) -> không khả thi; cách sinh: {Solution().primePalindrome(n)}")
t0 = time.perf_counter()
Solution().primePalindrome(n)
print(f"cách sinh chạy trong {1000 * (time.perf_counter() - t0):.0f} ms")

n = 10000005: duyệt tuần tự cần 90,029,997 bước (mỗi bước đảo số) -> không khả thi; cách sinh: 100030001
cách sinh chạy trong 1 ms


Vì sao chậm: số đối xứng 8 chữ số toàn hợp số, đáp án nằm sau khoảng đó. **Đúng:** nhảy tới `10^8` (Cách 2) hoặc sinh trực tiếp đối xứng độ dài lẻ (Cách 3-5).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "số đối xứng + điều kiện số học" -> **sinh các số đối xứng từ nửa trái** thay vì duyệt mọi số: giảm không gian từ `10^8` xuống `10^4`.
- **Ghi nhớ:** số đối xứng độ dài chẵn chia hết cho 11 (chỉ `11` nguyên tố).
- **Kiểm tra nguyên tố:** chia thử tới `sqrt` (`v * v <= x`) đủ cho `x <= 2 * 10^8`; Miller-Rabin cho `x` lớn hơn nhiều.
- **Sinh theo thứ tự tăng** cho phép dừng ở số đầu tiên thỏa.
- **Bảng tính sẵn** khi miền hữu hạn và số lời giải nhỏ (vài trăm).
- Cùng họ: Find the Closest Palindrome (564), Super Palindromes (906), Count Primes (204), Palindrome Number (9).
- Test nên có: `n = 1`, các độ dài chẵn (`8`, `12`, `1000`), mốc `10^7`, `10^8`.

## 15. Biến thể

1. **Số đối xứng nhỏ nhất `>= n`** (không cần nguyên tố): chọn nửa trái của `n`, dựng đối xứng, nếu nhỏ hơn `n` thì tăng nửa trái (chú ý tràn chữ số).
2. **Đếm số nguyên tố đối xứng trong `[a, b]`**: dùng bảng + `bisect` (bài tập 5).
3. **Số nguyên tố đối xứng thứ `k`**: chỉ số trong bảng.
4. Bài liên quan: Find the Closest Palindrome (564), Super Palindromes (906).

Dưới đây là code cho biến thể 1 và 2, kiểm bằng oracle.

In [32]:
def next_palindrome_ge(n: int) -> int:
    s = str(n)
    L = len(s)
    half = s[: (L + 1) // 2]
    def build(h):
        t = str(h)
        return int(t + (t[::-1] if L % 2 == 0 else t[-2::-1]))
    cand = build(int(half))
    if cand >= n:
        return cand
    h2 = int(half) + 1
    if len(str(h2)) > len(half):                         # tràn chữ số: 99 -> 100 -> đối xứng dài hơn (101 hoặc 1001)
        return 10 ** L + 1
    return build(h2)

def count_prime_palindromes(a: int, b: int) -> int:
    return bisect_left(TABLE, b + 1) - bisect_left(TABLE, a)

pals = set(all_palindromes_upto(10 ** 6))
sorted_pals = sorted(pals)
for n in range(1, 20000):
    assert next_palindrome_ge(n) == sorted_pals[bisect_left(sorted_pals, n)], n
rng = random.Random(15)
for _ in range(300):
    a = rng.randint(1, 10 ** 8)
    b = rng.randint(a, min(2 * 10 ** 8, a + rng.choice([10, 10 ** 4, 10 ** 7])))
    assert count_prime_palindromes(a, b) == sum(1 for x in oracle_table if a <= x <= b), (a, b)
print("số nguyên tố đối xứng trong [1, 10^4]:", count_prime_palindromes(1, 10 ** 4), "| trong [1, 10^8]:", count_prime_palindromes(1, 10 ** 8))
print("Biến thể: OK")

số nguyên tố đối xứng trong [1, 10^4]: 20 | trong [1, 10^8]: 781
Biến thể: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt tuần tự | Thử từng số | rất chậm với `n` lớn | `O(1)` | Hiểu đề; oracle với `n` nhỏ |
| 2. Duyệt có nhảy | Chặn `(10^7, 10^8)` | vài giây | `O(1)` | Lời giải chuẩn ngắn |
| 3. Sinh nửa trái + chia thử | Chỉ xét đối xứng độ dài lẻ | `~0,1-1 s` | `O(1)` | **Cách nộp chính** |
| 4. Sinh + Miller-Rabin | Kiểm tra nguyên tố nhanh | `~0,1 s` | `O(1)` | Khi giới hạn lớn hơn |
| 5. Bảng + `bisect` | Tính sẵn ~781 số | truy vấn `O(log)` | `O(781)` | Nhiều truy vấn |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
print(reverse_num(120), reverse_num(1221))
print([x for x in range(1, 20) if is_prime(x)])
print(Solution().primePalindrome(9))
```

In [33]:
# Bài 1: ghi dự đoán của bạn (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Liệt kê các số đối xứng 3 chữ số có nửa trái `10, 11, 12, 13` (dưới dạng list) và ghi số nào trong đó là nguyên tố.

In [34]:
# Bài 2: điền kết quả tính tay
hand_pals = None        # list 4 số đối xứng, theo thứ tự nửa trái 10, 11, 12, 13
hand_primes = None      # list các số nguyên tố trong hand_pals

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm kiểm tra nguyên tố và hàm đảo số.

In [35]:
def exercise_3_is_prime(x: int) -> bool:
    # TODO 1: x < 2 -> False
    # TODO 2: thử v = 2, 3, ... cho tới khi v * v > x; nếu x % v == 0 thì False
    # TODO 3: trả True
    return None


def exercise_3_reverse(x: int) -> int:
    rev = 0
    # TODO 4: while x: lấy chữ số cuối (x % 10), ghép vào rev (rev * 10 + chữ số), bỏ chữ số cuối (x //= 10)
    return None


# assert [x for x in range(1, 30) if exercise_3_is_prime(x)] == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
# assert exercise_3_is_prime(9) is False and exercise_3_is_prime(1) is False
# assert exercise_3_reverse(120) == 21 and exercise_3_reverse(12321) == 12321

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, hãy tự viết `exercise_4_generate(n)`: sinh số đối xứng độ dài lẻ (và `11`) từ nửa trái, trả số nguyên tố đầu tiên `>= n`.

In [36]:
def exercise_4_generate(n: int) -> int:
    # TODO
    pass


# assert exercise_4_generate(6) == 7
# assert exercise_4_generate(8) == 11
# assert exercise_4_generate(13) == 101
# assert exercise_4_generate(10 ** 7) == 100030001

### Bài 5 — Mini challenge: đếm và lấy số nguyên tố đối xứng thứ `k` (khó)

Viết `exercise_5_count(a, b)` đếm số nguyên tố đối xứng trong `[a, b]` (`1 <= a <= b <= 2 * 10^8`) và `exercise_5_kth(k)` trả số nguyên tố đối xứng thứ `k` (`k` tính từ 1: `2, 3, 5, 7, 11, 101, ...`). Không được duyệt mọi số trong `[a, b]`.

```python
exercise_5_count(1, 100)   -> 5      (2, 3, 5, 7, 11)
exercise_5_count(1, 200)   -> 10     (thêm 101, 131, 151, 181, 191)
exercise_5_kth(6)          -> 101
```
Gợi ý: sinh bảng như Cách 5 rồi `bisect`.

In [37]:
def exercise_5_count(a: int, b: int) -> int:
    # TODO
    pass


def exercise_5_kth(k: int) -> int:
    # TODO
    pass


# assert exercise_5_count(1, 100) == 5
# assert exercise_5_count(1, 200) == 10
# assert exercise_5_kth(6) == 101

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: đảo `120` được `021 = 21`; đảo `1221` được `1221`.
- Mức 2: nguyên tố nhỏ hơn 20: `2, 3, 5, 7, 11, 13, 17, 19`; `9` đối xứng nhưng không nguyên tố.
- Mức 3: `21 1221`, danh sách trên, và `11`.

**Bài 2**
- Mức 1: nửa trái `10` -> `101` (ghép `10` với `1` đảo `s[-2::-1]` = `"1"`).
- Mức 2: `11 -> 111`, `12 -> 121`, `13 -> 131`.
- Mức 3: `[101, 111, 121, 131]`; nguyên tố: `[101, 131]` (`111 = 3 * 37`, `121 = 11^2`).

**Bài 3**
- Mức 1: vòng `while v * v <= x`.
- Mức 2: đảo số: `rev = rev * 10 + x % 10; x //= 10`.
- Mức 3: `return True` ở cuối; `return rev`.

**Bài 4**
- Mức 1: `for L in range(len(str(n)), 10)`; bỏ độ dài chẵn trừ 2.
- Mức 2: nửa trái `h` từ `10 ** (half - 1)` tới `10 ** half - 1`; `pal = int(s + s[-2::-1])` (lẻ).
- Mức 3: `if pal >= n and is_prime(pal): return pal`.

**Bài 5**
- Mức 1: dựng bảng như Cách 5.
- Mức 2: đếm: `bisect_left(table, b + 1) - bisect_left(table, a)`.
- Mức 3: `table[k - 1]`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [38]:
print(reverse_num(120), reverse_num(1221))
print([x for x in range(1, 20) if is_prime(x)])
print(Solution().primePalindrome(9))

assert (reverse_num(120), reverse_num(1221)) == (21, 1221)
assert [x for x in range(1, 20) if is_prime(x)] == [2, 3, 5, 7, 11, 13, 17, 19]
assert Solution().primePalindrome(9) == 11

21 1221
[2, 3, 5, 7, 11, 13, 17, 19]
11


### Đáp án Bài 2

Nửa trái `10, 11, 12, 13` cho `101, 111, 121, 131`. `111 = 3 * 37` và `121 = 11 * 11` là hợp số. Nguyên tố: `101` và `131`.

In [39]:
hand_pals_solution = [101, 111, 121, 131]
hand_primes_solution = [101, 131]
pals = [int(str(h) + str(h)[-2::-1]) for h in (10, 11, 12, 13)]
assert pals == hand_pals_solution
assert [x for x in pals if is_prime(x)] == hand_primes_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [40]:
def exercise_3_is_prime_solution(x: int) -> bool:
    if x < 2:
        return False
    v = 2
    while v * v <= x:
        if x % v == 0:
            return False
        v += 1
    return True

def exercise_3_reverse_solution(x: int) -> int:
    rev = 0
    while x:
        rev = rev * 10 + x % 10
        x //= 10
    return rev

assert [x for x in range(1, 30) if exercise_3_is_prime_solution(x)] == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
assert exercise_3_is_prime_solution(9) is False and exercise_3_is_prime_solution(1) is False
assert exercise_3_reverse_solution(120) == 21 and exercise_3_reverse_solution(12321) == 12321
assert all(exercise_3_is_prime_solution(x) == is_prime(x) for x in range(0, 5000))
assert all(exercise_3_reverse_solution(x) == int(str(x)[::-1]) for x in range(0, 5000))
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [41]:
def exercise_4_generate_solution(n: int) -> int:
    for L in range(len(str(n)), 10):
        if L % 2 == 0 and L != 2:
            continue
        half = (L + 1) // 2
        for h in range(10 ** (half - 1), 10 ** half):
            s = str(h)
            pal = int(s + (s[::-1] if L % 2 == 0 else s[-2::-1]))
            if pal >= n and is_prime(pal):
                return pal
    return -1

assert exercise_4_generate_solution(6) == 7
assert exercise_4_generate_solution(8) == 11
assert exercise_4_generate_solution(13) == 101
assert exercise_4_generate_solution(10 ** 7) == 100030001
rng = random.Random(4)
for _ in range(200):
    n = rng.randint(1, 10 ** 8)
    assert exercise_4_generate_solution(n) == oracle(n), n
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [42]:
def exercise_5_count_solution(a: int, b: int) -> int:
    return bisect_left(TABLE, b + 1) - bisect_left(TABLE, a)

def exercise_5_kth_solution(k: int) -> int:
    return TABLE[k - 1]

assert exercise_5_count_solution(1, 100) == 5
assert exercise_5_count_solution(1, 200) == 10
assert exercise_5_kth_solution(6) == 101
rng = random.Random(5)
for _ in range(200):
    a = rng.randint(1, 10 ** 8)
    b = rng.randint(a, 2 * 10 ** 8)
    assert exercise_5_count_solution(a, b) == sum(1 for x in oracle_table if a <= x <= b)
assert [exercise_5_kth_solution(k) for k in range(1, 12)] == oracle_table[:11]
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (sinh số đối xứng từ nửa trái)

```python
class Solution:
    def primePalindrome(self, n: int) -> int:
        def is_prime(x: int) -> bool:
            if x < 2:
                return False
            v = 2
            while v * v <= x:                         # có dấu = (vd 9)
                if x % v == 0:
                    return False
                v += 1
            return True

        for L in range(len(str(n)), 10):              # độ dài tăng dần
            if L % 2 == 0 and L != 2:                 # đối xứng chẵn chia hết 11 (trừ chính 11)
                continue
            half = (L + 1) // 2
            for h in range(10 ** (half - 1), 10 ** half):
                s = str(h)
                pal = int(s + (s[::-1] if L % 2 == 0 else s[-2::-1]))
                if pal >= n and is_prime(pal):
                    return pal
        return -1
```

Công thức nhớ nhanh: đối xứng chẵn (trừ 11) là hợp số; sinh nửa trái `h` -> `s + s[-2::-1]` (lẻ) / `s + s[::-1]` (chẵn).

### Checklist trước khi submit

- [ ] `is_prime` xử lý `x < 2` và dùng `v * v <= x`?
- [ ] Không bỏ sót `11` (độ dài 2 được giữ)?
- [ ] Bỏ qua độ dài chẵn khác 2 (hoặc xử lý khoảng `(10^7, 10^8)`)?
- [ ] Số đối xứng độ dài lẻ dựng bằng `s + s[-2::-1]`?
- [ ] Vòng độ dài bắt đầu từ `len(str(n))` và dừng ở 9?
- [ ] Đã thử: `n = 1, 8, 12, 999, 10^7, 10^8`?
- [ ] Tên method đúng `primePalindrome(self, n)` và trả `int`?